# Training Hub JIT checkpointing on Kubeflow (RHOAIENG-83586)

Submits a `TrainingHubTrainer` job with `enable_jit_checkpoint=True` passed through `func_args`, suspends the TrainJob mid-run (`spec.suspend=True`, which sends SIGTERM to its pods), checks the checkpoint on the shared PVC, then resumes the same TrainJob (`spec.suspend=False`) and checks that training continues from the checkpoint.

Parameters (env vars):
- `JIT_ALGORITHM`: `lora_sft` | `sft` | `osft`
- `JIT_NUM_GPUS`: GPUs on the single training node (default 1)
- `JIT_SCENARIO`:
  - `resume`: SIGTERM saves a checkpoint and the resumed TrainJob continues from it
  - `incomplete` (`lora_sft` only): a checkpoint with an incomplete marker is not used for resume


In [ ]:
import json
import os
import re
import shutil
import sys
import time
from pathlib import Path

ALGORITHM = os.environ.get("JIT_ALGORITHM", "lora_sft")
SCENARIO = os.environ.get("JIT_SCENARIO", "resume")
NUM_GPUS = int(os.environ.get("JIT_NUM_GPUS", "1"))  # GPUs on the single training node
if ALGORITHM not in {"lora_sft", "sft", "osft"}:
    raise ValueError(f"Unsupported JIT_ALGORITHM: {ALGORITHM}")
if SCENARIO not in {"resume", "incomplete"}:
    raise ValueError(f"Unsupported JIT_SCENARIO: {SCENARIO}")
if SCENARIO == "incomplete" and ALGORITHM != "lora_sft":
    raise ValueError("The incomplete scenario uses the HF checkpoint layout (lora_sft only)")

PVC_ROOT = Path("/opt/app-root/src")
WORK_DIR = PVC_ROOT / "jit-e2e"
CKPT_DIR = WORK_DIR / f"ckpt-{ALGORITHM}-{SCENARIO}"
DATA_PATH = WORK_DIR / "data.jsonl"
MODEL_DIR = PVC_ROOT / "models" / "Qwen2.5-0.5B-Instruct"

# 400 rows at batch 4 per GPU is about 100 steps per epoch on 1 GPU.
NUM_ROWS = 400
LORA_MICRO_BATCH = 4
LORA_STEPS_PER_EPOCH = NUM_ROWS // (LORA_MICRO_BATCH * NUM_GPUS)
if ALGORITHM == "lora_sft":
    # 3 epochs with a periodic checkpoint at the end of each epoch; SIGTERM in
    # epoch 2, so the JIT checkpoint must be newer than the epoch-1 one.
    NUM_EPOCHS = 3
    KILL_AFTER_STEP = LORA_STEPS_PER_EPOCH + 5
else:
    # No periodic checkpoints, so any checkpoint found came from JIT.
    NUM_EPOCHS = 2
    KILL_AFTER_STEP = 5

print(f"algorithm={ALGORITHM} scenario={SCENARIO} gpus={NUM_GPUS} ckpt_dir={CKPT_DIR}")


In [ ]:
from kubernetes import client as k8s, config as k8s_config

_ns_file = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
NAMESPACE = (
    open(_ns_file).read().strip()
    if os.path.exists(_ns_file)
    else os.environ.get("NOTEBOOK_NAMESPACE", "")
)

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if api_server and token:
    configuration = k8s.Configuration()
    configuration.host = api_server
    configuration.verify_ssl = False
    configuration.api_key = {"authorization": f"Bearer {token}"}
    api_client = k8s.ApiClient(configuration)
else:
    k8s_config.load_incluster_config()
    api_client = k8s.ApiClient()

PVC_NAME = os.environ["SHARED_PVC_NAME"]
core = k8s.CoreV1Api(api_client)
print(f"Namespace: {NAMESPACE!r}, PVC: {PVC_NAME}")


In [ ]:
# Fresh checkpoint dir and a small chat dataset on the shared PVC.
if CKPT_DIR.exists():
    shutil.rmtree(CKPT_DIR)
CKPT_DIR.mkdir(parents=True)

QA = [
    ("What is 2+2?", "4"),
    ("Capital of France?", "Paris"),
    ("Say hello", "Hello!"),
    ("Color of the sky?", "Blue"),
    ("What is 3+5?", "8"),
    ("Opposite of hot?", "Cold"),
    ("Largest planet?", "Jupiter"),
    ("How many legs does a cat have?", "Four"),
]
with DATA_PATH.open("w") as f:
    for i in range(NUM_ROWS):
        question, answer = QA[i % len(QA)]
        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        f.write(json.dumps({"messages": messages}) + "\n")
print(f"Wrote {NUM_ROWS} rows to {DATA_PATH}")


In [ ]:
# Model: S3/MinIO (disconnected) when configured, otherwise HuggingFace.
def fetch_model_from_s3() -> bool:
    endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
    bucket = os.getenv("AWS_STORAGE_BUCKET", "")
    if not (endpoint and bucket):
        return False
    import s3fs

    url = endpoint if endpoint.startswith("http") else f"https://{endpoint}"
    fs = s3fs.S3FileSystem(
        key=os.getenv("AWS_ACCESS_KEY_ID", ""),
        secret=os.getenv("AWS_SECRET_ACCESS_KEY", ""),
        endpoint_url=url,
        client_kwargs={"verify": False},
    )
    prefix = os.getenv("AWS_STORAGE_BUCKET_JIT_DIR", "").strip("/")
    remote = "/".join(p for p in (bucket, prefix, MODEL_DIR.name) if p)
    if not fs.exists(remote):
        print(f"Model not found at s3://{remote}")
        return False
    fs.get(remote + "/", str(MODEL_DIR) + "/", recursive=True)
    print(f"Model downloaded from s3://{remote}")
    return True


if not (MODEL_DIR / "config.json").exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    if not fetch_model_from_s3():
        from huggingface_hub import snapshot_download

        snapshot_download(
            repo_id="Qwen/Qwen2.5-0.5B-Instruct",
            local_dir=str(MODEL_DIR),
            token=os.getenv("HUGGINGFACE_HUB_TOKEN"),
        )
        print(f"Model downloaded from HuggingFace to {MODEL_DIR}")
print(f"Model ready: {MODEL_DIR}")


In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer

client = TrainerClient(KubernetesBackendConfig(client_configuration=api_client.configuration))

runtime_name = os.getenv("TRAINING_RUNTIME", "training-hub")
th_runtime = next((r for r in client.list_runtimes() if r.name == runtime_name), None)
if th_runtime is None:
    raise RuntimeError(f"Runtime {runtime_name!r} not found")
print(f"Using runtime: {th_runtime.name}")


In [ ]:
# The callback class must be importable from a file so kubeflow-sdk can
# serialize it with inspect.getsource.
sys.path.insert(0, str(PVC_ROOT))
from jit_step_probe import JITStepProbe

print(f"Callback ready: {JITStepProbe.__name__}")


In [ ]:
from kubeflow.trainer.options import (
    ContainerPatch,
    JobSetSpecPatch,
    JobSetTemplatePatch,
    JobSpecPatch,
    JobTemplatePatch,
    PodSpecPatch,
    PodTemplatePatch,
    ReplicatedJobPatch,
    RuntimePatch,
    TrainingRuntimeSpecPatch,
)

ALGORITHMS = {
    "lora_sft": TrainingHubAlgorithms.LORA_SFT,
    "sft": TrainingHubAlgorithms.SFT,
    "osft": TrainingHubAlgorithms.OSFT,
}
STEP_RE = re.compile(r"\[JIT-E2E\] step=(\d+) epoch=([\d.]+)")
# Log line each backend prints when it loads a checkpoint, with the saved step.
RESUME_RE = {
    "lora_sft": re.compile(r"Resuming training from checkpoint: \S*checkpoint-(\d+)"),
    "sft": re.compile(r"Resuming mid-epoch: epoch=\d+, last_step=(\d+)"),
    "osft": re.compile(r"Resumed at step=(\d+)"),
}

PVC_PATCH = RuntimePatch(
    training_runtime_spec=TrainingRuntimeSpecPatch(
        template=JobSetTemplatePatch(
            spec=JobSetSpecPatch(
                replicated_jobs=[
                    ReplicatedJobPatch(
                        name="node",
                        template=JobTemplatePatch(
                            spec=JobSpecPatch(
                                template=PodTemplatePatch(
                                    spec=PodSpecPatch(
                                        volumes=[
                                            {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                                        ],
                                        containers=[
                                            ContainerPatch(
                                                name="node",
                                                volume_mounts=[
                                                    {"name": "work", "mountPath": str(PVC_ROOT), "readOnly": False},
                                                ],
                                            )
                                        ],
                                    )
                                )
                            )
                        ),
                    )
                ]
            )
        )
    )
)


def build_params(num_epochs: int) -> dict:
    # JIT is a Training Hub feature; the SDK passes these through func_args.
    params = {
        "model_path": str(MODEL_DIR),
        "data_path": str(DATA_PATH),
        "ckpt_output_dir": str(CKPT_DIR),
        "num_epochs": num_epochs,
        "max_seq_len": 128,
        "enable_jit_checkpoint": True,
        "nnodes": 1,
        "nproc_per_node": NUM_GPUS,
    }
    if ALGORITHM == "lora_sft":
        params.update(
            micro_batch_size=LORA_MICRO_BATCH,
            gradient_accumulation_steps=1,
            learning_rate=2e-4,
            lora_r=8,
            lora_alpha=16,
            sample_packing=False,
            logging_steps=1,
            save_steps=LORA_STEPS_PER_EPOCH,  # periodic checkpoint at each epoch end
        )
    elif ALGORITHM == "sft":
        params.update(
            effective_batch_size=4,
            learning_rate=2e-5,
            save_samples=0,
            checkpoint_at_epoch=False,
        )
    else:
        params.update(
            effective_batch_size=4,
            learning_rate=2e-5,
            unfreeze_rank_ratio=0.1,
            max_tokens_per_gpu=2048,
            checkpoint_at_epoch=False,
        )
    return params


def submit(num_epochs: int = NUM_EPOCHS) -> str:
    job_name = client.train(
        trainer=TrainingHubTrainer(
            algorithm=ALGORITHMS[ALGORITHM],
            func_args=build_params(num_epochs),
            # The runtime image ships training-hub 0.10.0, which predates JIT.
            packages_to_install=["training-hub>=0.11.0"],
            env={
                "HF_HOME": str(PVC_ROOT / ".cache" / "huggingface"),
                "TRITON_CACHE_DIR": "/tmp/.triton",
                "TOKENIZERS_PARALLELISM": "false",
            },
            callbacks=[JITStepProbe],
            resources_per_node={"cpu": 4, "memory": "32Gi", "nvidia.com/gpu": NUM_GPUS},
        ),
        options=[PVC_PATCH],
        runtime=th_runtime,
    )
    print(f"Submitted {job_name} ({ALGORITHM}, {num_epochs} epochs, {NUM_GPUS} GPU(s))")
    return job_name


def job_log_text(job_name: str) -> str:
    try:
        return "\n".join(str(line) for line in client.get_job_logs(name=job_name, follow=False))
    except Exception as exc:  # pod not started yet
        print(f"logs not available yet: {exc}")
        return ""


def logged_steps(text: str) -> list[tuple[int, float]]:
    return [(int(step), float(epoch)) for step, epoch in STEP_RE.findall(text)]


def wait_for_step(job_name: str, min_step: int, timeout: int = 1800) -> tuple[int, float]:
    deadline = time.time() + timeout
    while time.time() < deadline:
        status = client.get_job(name=job_name).status
        if status in ("Complete", "Failed"):
            print(job_log_text(job_name)[-5000:])
            raise RuntimeError(f"{job_name} ended ({status}) before reaching step {min_step}")
        steps = logged_steps(job_log_text(job_name))
        if steps and steps[-1][0] >= min_step:
            return steps[-1]
        time.sleep(3)
    raise TimeoutError(f"{job_name} did not reach step {min_step} in {timeout}s")


def training_pods(job_name: str) -> list:
    selector = f"jobset.sigs.k8s.io/jobset-name={job_name}"
    return core.list_namespaced_pod(NAMESPACE, label_selector=selector).items


TRAINJOB = {"group": "trainer.kubeflow.org", "version": "v1alpha1", "plural": "trainjobs"}
custom = k8s.CustomObjectsApi(api_client)


def set_suspend(job_name: str, suspend: bool) -> None:
    body = {"spec": {"suspend": suspend}}
    try:
        custom.patch_namespaced_custom_object(
            namespace=NAMESPACE, name=job_name, body=body,
            _content_type="application/merge-patch+json", **TRAINJOB,
        )
    except TypeError:  # older clients send a merge patch for dict bodies by default
        custom.patch_namespaced_custom_object(namespace=NAMESPACE, name=job_name, body=body, **TRAINJOB)
    print(f"{job_name}: spec.suspend={suspend}")


def trainjob_condition(job_name: str, condition_type: str) -> str | None:
    job = custom.get_namespaced_custom_object(namespace=NAMESPACE, name=job_name, **TRAINJOB)
    for condition in (job.get("status") or {}).get("conditions") or []:
        if condition.get("type") == condition_type:
            return condition.get("status")
    return None


def suspend_and_wait(job_name: str, timeout: int = 600) -> None:
    # Suspending deletes the training pods; each gets SIGTERM and JIT saves
    # within the pod's termination grace period.
    set_suspend(job_name, True)
    deadline = time.time() + timeout
    while training_pods(job_name):
        if time.time() > deadline:
            raise TimeoutError(f"Pods of {job_name} did not terminate after suspend")
        time.sleep(3)
    print(f"{job_name}: training pods terminated")


def wait_done(job_name: str, timeout: int = 3600) -> tuple[str, str]:
    client.wait_for_job_status(name=job_name, status={"Complete", "Failed"}, timeout=timeout)
    return client.get_job(name=job_name).status, job_log_text(job_name)


def checkpoint_step(path: Path) -> int:
    return int(re.findall(r"\d+", path.name)[-1])


def jit_checkpoints() -> list[Path]:
    if ALGORITHM == "lora_sft":
        return sorted(
            (
                p
                for p in CKPT_DIR.glob("checkpoint-*")
                if p.is_dir() and not (CKPT_DIR / f".incomplete-checkpoint-{checkpoint_step(p)}").exists()
            ),
            key=checkpoint_step,
        )
    if ALGORITHM == "sft":
        return sorted(p.parent for p in CKPT_DIR.glob("full_state/epoch_*/training_metadata.json"))
    return sorted(p.parent for p in CKPT_DIR.glob("full_state_checkpoints/step_*/training_state.pt"))


results: list[tuple[bool, str]] = []


def check(ok: bool, message: str) -> None:
    print(f"[{'PASS' if ok else 'FAIL'}] {message}", flush=True)
    results.append((ok, message))


In [ ]:
if SCENARIO == "resume":
    job = submit()
    step, epoch = wait_for_step(job, KILL_AFTER_STEP)
    print(f"Reached step {step} (epoch index {epoch}); suspending the TrainJob")

    before = jit_checkpoints()
    if ALGORITHM == "lora_sft":
        check(epoch >= 1, f"SIGTERM is sent during epoch 2 (epoch index {epoch})")
        check(
            [p.name for p in before] == [f"checkpoint-{LORA_STEPS_PER_EPOCH}"],
            f"only the epoch-1 checkpoint exists before SIGTERM: {[p.name for p in before]}",
        )
    else:
        check(not before, f"no checkpoint exists before SIGTERM: {[p.name for p in before]}")

    suspend_and_wait(job)
    check(trainjob_condition(job, "Suspended") == "True", "TrainJob is suspended, not completed or failed")
    new = [p for p in jit_checkpoints() if p not in before]
    check(bool(new), f"JIT checkpoint saved on SIGTERM: {[p.name for p in new]}")

    set_suspend(job, False)
    status, text = wait_done(job)
    match = RESUME_RE[ALGORITHM].search(text)
    check(match is not None, "resumed TrainJob loaded the JIT checkpoint without a resume argument")
    if ALGORITHM == "lora_sft" and match and new:
        check(
            int(match.group(1)) == checkpoint_step(new[-1]),
            f"resumed from the JIT checkpoint {new[-1].name}, not the epoch-1 one",
        )
    steps = logged_steps(text)
    if match and steps:
        resumed_at = int(match.group(1))
        check(steps[0][0] == resumed_at + 1, f"first step after resume is {resumed_at + 1} (got {steps[0][0]})")
        check(steps[-1][0] > resumed_at, f"training continued past step {resumed_at} (last step {steps[-1][0]})")
    check(status == "Complete", f"resumed TrainJob status is Complete (got {status})")
    if ALGORITHM == "osft":
        check("is_main_process" not in text, "no is_main_process error in osft logs")
    client.delete_job(job)

else:  # incomplete
    fake = CKPT_DIR / "checkpoint-999"
    fake.mkdir()
    (CKPT_DIR / ".incomplete-checkpoint-999").touch()
    job = submit(num_epochs=1)
    status, text = wait_done(job)
    check(RESUME_RE[ALGORITHM].search(text) is None, "incomplete checkpoint-999 was not used for resume")
    steps = logged_steps(text)
    check(bool(steps) and steps[0][0] == 1, f"training started at step 1 (got {steps[0][0] if steps else None})")
    check(status == "Complete", f"job status is Complete (got {status})")
    client.delete_job(job)


In [ ]:
failed = [message for ok, message in results if not ok]
if failed:
    raise AssertionError(f"JIT e2e ({ALGORITHM}/{SCENARIO}) failed: {failed}")
print(f"JIT_E2E_RESULT: PASS ({ALGORITHM}/{SCENARIO})")
